# Football Player Market Value Analysis  
## Step 1 - Data Profiling

This notebook performs only **Step 1 of the Data Preparation phase: Data Profiling** for football player market value analysis.  
The focus is to understand the available datasets, inspect metadata, profile key variables, and evaluate each dataset's relevance to the research question.


# 1 Project Goal

The research goal is to analyze football player market values and identify factors that influence player valuation and transfers.

Core question:

**What factors influence football player market value?**

In this step, we only profile available data sources to assess their analytical potential before any structuring, enrichment, or cleaning.


# 2 Data Sources Overview

We profile three local data sources:

- **Dataset A - Transfermarkt Football Dataset** (`./football-datasets`)  
  Large multi-table Transfermarkt-style datalake with player profiles, market values, performances, transfers, injuries, and team-level context.

- **Dataset B - European Football Transfers Dataset** (`./football-transfers-data/dataset/transfers.csv`)  
  Transfer records across major European leagues with transfer fee, market value at transfer, player details, and transfer timing.

- **Dataset C - Soccer Player Value Modeling Dataset Repository** (`./Modelling-Football-Players-Values-on-a-Transfer-Market`)  
  Modeling notebooks and scraper code used for player value prediction workflows. This repository may include local CSVs or references to externally sourced files.


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('missingno') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'missingno'])

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno

from IPython.display import display

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 10

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 120)


In [ ]:
ROOT = Path('.').resolve()
DATASET_A_ROOT = ROOT / 'football-datasets'
DATASET_B_FILE = ROOT / 'football-transfers-data' / 'dataset' / 'transfers.csv'
DATASET_C_ROOT = ROOT / 'Modelling-Football-Players-Values-on-a-Transfer-Market'


def size_mb(path: Path) -> float:
    return round(path.stat().st_size / (1024 * 1024), 2)


def discover_dataset_c_csv_references(dataset_c_root: Path):
    refs = set()
    for nb_file in dataset_c_root.rglob('*.ipynb'):
        try:
            nb = json.loads(nb_file.read_text(encoding='utf-8', errors='ignore'))
        except Exception:
            continue
        for cell in nb.get('cells', []):
            if cell.get('cell_type') != 'code':
                continue
            src = ''.join(cell.get('source', []))
            for m in re.findall(r'read_csv\((?:r)?[\'"]([^\'"]+\.csv)[\'"]', src):
                refs.add(m)
    return sorted(refs)


# Dataset source inventory

dataset_a_csvs = sorted((DATASET_A_ROOT / 'datalake' / 'transfermarkt').rglob('*.csv'))
dataset_c_csvs = sorted(DATASET_C_ROOT.rglob('*.csv'))
dataset_c_expected_csv_refs = discover_dataset_c_csv_references(DATASET_C_ROOT)

sources_overview = pd.DataFrame([
    {
        'dataset': 'Dataset A - Transfermarkt Football Dataset',
        'local_path': str(DATASET_A_ROOT),
        'tabular_files_found': len(dataset_a_csvs),
        'size_of_main_file_mb': max([size_mb(p) for p in dataset_a_csvs]) if dataset_a_csvs else np.nan,
        'example_files': ', '.join([p.name for p in dataset_a_csvs[:5]])
    },
    {
        'dataset': 'Dataset B - European Football Transfers Dataset',
        'local_path': str(DATASET_B_FILE),
        'tabular_files_found': int(DATASET_B_FILE.exists()),
        'size_of_main_file_mb': size_mb(DATASET_B_FILE) if DATASET_B_FILE.exists() else np.nan,
        'example_files': DATASET_B_FILE.name if DATASET_B_FILE.exists() else 'missing'
    },
    {
        'dataset': 'Dataset C - Soccer Player Value Modeling Dataset',
        'local_path': str(DATASET_C_ROOT),
        'tabular_files_found': len(dataset_c_csvs),
        'size_of_main_file_mb': max([size_mb(p) for p in dataset_c_csvs]) if dataset_c_csvs else np.nan,
        'example_files': ', '.join([p.name for p in dataset_c_csvs[:5]]) if dataset_c_csvs else 'No local CSV found'
    },
])

print('Data source inventory:')
display(sources_overview)

if dataset_c_expected_csv_refs:
    print('Dataset C notebooks reference these CSV file names:')
    display(pd.DataFrame({'referenced_csv': dataset_c_expected_csv_refs}))


# 3 Data Loading

Load representative tables from Dataset A, the transfer table from Dataset B, and any discoverable CSVs from Dataset C.

For each loaded table, print:

- shape
- first rows
- column names


In [ ]:
def load_csv(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')
    if 'low_memory' not in kwargs:
        kwargs['low_memory'] = False
    return pd.read_csv(path, **kwargs)


representative_files = {
    'Dataset A - player_profiles': DATASET_A_ROOT / 'datalake' / 'transfermarkt' / 'player_profiles' / 'player_profiles.csv',
    'Dataset A - player_market_value': DATASET_A_ROOT / 'datalake' / 'transfermarkt' / 'player_market_value' / 'player_market_value.csv',
    'Dataset A - player_performances': DATASET_A_ROOT / 'datalake' / 'transfermarkt' / 'player_performances' / 'player_performances.csv',
    'Dataset A - transfer_history': DATASET_A_ROOT / 'datalake' / 'transfermarkt' / 'transfer_history' / 'transfer_history.csv',
    'Dataset B - transfers': DATASET_B_FILE,
}

loaded_datasets = {}

for name, path in representative_files.items():
    df = load_csv(path)
    loaded_datasets[name] = {'df': df, 'path': path}
    print(f'\n{name}')
    print(f'Path: {path}')
    print(f'Shape: {df.shape}')
    print('Columns:')
    print(df.columns.tolist())
    display(df.head(3))

# Dataset C handling
if dataset_c_csvs:
    for idx, c_path in enumerate(dataset_c_csvs[:3], start=1):
        cname = f'Dataset C - file_{idx} ({c_path.name})'
        cdf = load_csv(c_path)
        loaded_datasets[cname] = {'df': cdf, 'path': c_path}
        print(f'\n{cname}')
        print(f'Path: {c_path}')
        print(f'Shape: {cdf.shape}')
        print('Columns:')
        print(cdf.columns.tolist())
        display(cdf.head(3))
else:
    print('\nDataset C: no local CSV files found to load directly.')
    # Create a small profiling table from notebook CSV references so Dataset C is still profiled at repository level.
    c_ref_rows = []
    for ref in dataset_c_expected_csv_refs:
        candidates = list(ROOT.rglob(Path(ref).name))
        c_ref_rows.append({
            'referenced_csv_name': Path(ref).name,
            'referenced_path_in_notebook': ref,
            'exists_somewhere_in_workspace': bool(candidates),
            'workspace_match_example': str(candidates[0]) if candidates else np.nan,
        })
    dataset_c_reference_df = pd.DataFrame(c_ref_rows)
    loaded_datasets['Dataset C - referenced_csv_inventory'] = {
        'df': dataset_c_reference_df,
        'path': DATASET_C_ROOT,
    }
    print('Created profiling table from Dataset C notebook CSV references.')
    print(f'Shape: {dataset_c_reference_df.shape}')
    display(dataset_c_reference_df.head(10))


# 4 Metadata Extraction

For each loaded table:

- inspect schema with `df.info()`
- inspect summary stats with `df.describe()`
- compute missing values with `df.isnull().sum()`

Then create a metadata summary dataframe containing dataset-level metadata.


In [ ]:
def dataset_metadata_row(dataset_name: str, path: Path, df: pd.DataFrame) -> dict:
    dtypes_map = {col: str(dtype) for col, dtype in df.dtypes.items()}
    missing_by_col = df.isnull().sum().to_dict()
    return {
        'dataset_name': dataset_name,
        'source_path': str(path),
        'n_rows': int(df.shape[0]),
        'n_columns': int(df.shape[1]),
        'column_names': list(df.columns),
        'data_types': dtypes_map,
        'missing_values_total': int(df.isnull().sum().sum()),
        'missing_values_by_column': missing_by_col,
    }


metadata_rows = []

for dname, payload in loaded_datasets.items():
    df = payload['df']
    path = payload['path']

    print(f'\n==================== {dname} ====================')
    print(f'Path: {path}')
    print(f'Shape: {df.shape}')

    print('\n[info]')
    df.info()

    print('\n[describe - numeric]')
    numeric_cols = df.select_dtypes(include=np.number).columns
    if len(numeric_cols) > 0:
        display(df[numeric_cols].describe().T)
    else:
        print('No numeric columns found.')

    print('\n[describe - object/categorical]')
    obj_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    if len(obj_cols) > 0:
        display(df[obj_cols].describe().T)
    else:
        print('No object/categorical columns found.')

    print('\n[missing values by column - top 15]')
    missing_series = df.isnull().sum().sort_values(ascending=False)
    display(missing_series.head(15).to_frame('missing_count'))

    metadata_rows.append(dataset_metadata_row(dname, path, df))

metadata_df = pd.DataFrame(metadata_rows)

print('\nMetadata summary table:')
display(metadata_df[['dataset_name', 'n_rows', 'n_columns', 'missing_values_total']])


# 5 Data Profiling

Perform exploratory profiling for each loaded table with:

- value distributions
- missing value analysis
- basic statistics
- categorical frequency analysis

Visualizations included:

- missing value heatmap (`missingno`)
- histograms for numeric variables
- bar charts for top categorical frequencies


In [ ]:
def profile_dataset(df: pd.DataFrame, dataset_name: str, sample_rows: int = 5000):
    print(f'\n----- Profiling: {dataset_name} -----')

    if df.empty:
        print('DataFrame is empty. Skipping profiling visuals.')
        return

    # Sampling only for plotting speed; tabular stats are based on full df where shown.
    sample_n = min(sample_rows, len(df))
    sample_df = df.sample(sample_n, random_state=42) if len(df) > sample_n else df.copy()

    # Missing values analysis
    missing_ratio = (df.isnull().mean() * 100).sort_values(ascending=False)
    print('Top columns by missing ratio (%):')
    display(missing_ratio.head(10).round(2).to_frame('missing_pct'))

    # Missing value heatmap/matrix
    heatmap_cols = sample_df.columns[: min(40, sample_df.shape[1])]
    msno.matrix(sample_df[heatmap_cols], figsize=(12, 4), sparkline=False)
    plt.title(f'{dataset_name} - Missing Value Matrix (sample rows, first columns)')
    plt.show()

    # Numeric value distributions
    numeric_cols = list(sample_df.select_dtypes(include=np.number).columns)
    if numeric_cols:
        cols_to_plot = numeric_cols[: min(6, len(numeric_cols))]
        fig, axes = plt.subplots(1, len(cols_to_plot), figsize=(5 * len(cols_to_plot), 3.5))
        if len(cols_to_plot) == 1:
            axes = [axes]
        for ax, col in zip(axes, cols_to_plot):
            sns.histplot(sample_df[col].dropna(), bins=30, kde=True, ax=ax, color='#1f77b4')
            ax.set_title(f'{col} distribution')
            ax.set_xlabel(col)
        plt.tight_layout()
        plt.show()
    else:
        print('No numeric columns available for histogram profiling.')

    # Basic statistics
    if numeric_cols:
        print('Basic statistics (numeric columns):')
        display(df[numeric_cols].describe().T)

    # Categorical frequency analysis
    cat_cols = list(sample_df.select_dtypes(include=['object', 'category', 'bool']).columns)
    if cat_cols:
        cols_to_plot = cat_cols[: min(4, len(cat_cols))]
        for col in cols_to_plot:
            vc = sample_df[col].astype(str).replace('nan', np.nan).dropna().value_counts().head(10)
            if vc.empty:
                continue
            plt.figure(figsize=(9, 3.5))
            sns.barplot(x=vc.values, y=vc.index, palette='viridis')
            plt.title(f'{dataset_name} - Top categories in {col}')
            plt.xlabel('Count')
            plt.ylabel(col)
            plt.tight_layout()
            plt.show()

            print(f'Top frequency table for {col}:')
            display(vc.to_frame('count'))
    else:
        print('No categorical columns available for frequency profiling.')


for dname, payload in loaded_datasets.items():
    profile_dataset(payload['df'], dname)


# 6 Dataset Relevance to Research Question

## Dataset A - Transfermarkt Football Dataset
This dataset is highly relevant because it directly includes market value signals and explanatory factors:

- market value history (`player_market_value`, `player_latest_market_value`)
- player profile attributes (age proxy from birth date, position, footedness, club context)
- performance indicators (goals, assists, minutes, cards, clean sheets)
- transfer history and transfer fee descriptors
- injury history and national/team context

Contribution to research question: provides both target-like information (market value) and multiple potential drivers of value.

## Dataset B - European Football Transfers Dataset
This dataset is highly relevant for transfer economics and timing:

- transfer fee amount (`transfer_fee_amnt`)
- market value amount at transfer (`market_val_amnt`)
- transfer window/season, league, direction (`in`/`left`)
- player age, nationality, position

Contribution to research question: supports analysis of how observed transfer behavior and transfer fee relate to market valuation.

## Dataset C - Soccer Player Value Modeling Dataset Repository
This repository is methodologically relevant:

- includes prior modeling notebooks and feature engineering logic for value prediction
- references modeling datasets (CSV files) used in experiments
- may or may not contain local raw CSVs in the cloned workspace

Contribution to research question: helps identify candidate modeling features and benchmark approaches. Even when local CSVs are absent, notebook references indicate expected modeling data structures.

Combining A + B + C is useful because A provides rich player history, B provides explicit transfer-market transactions, and C provides modeling context for feature selection and prediction strategy.


# 7 Preliminary Dataset Comparison


In [ ]:
comparison_rows = []

for dname, payload in loaded_datasets.items():
    df = payload['df']
    cols_lower = [c.lower() for c in df.columns]

    key_vars = []
    for kw in ['value', 'market', 'transfer', 'fee', 'age', 'position', 'goal', 'assist', 'minute', 'injury']:
        key_vars.extend([c for c in df.columns if kw in c.lower()])
    key_vars = sorted(set(key_vars))[:12]

    if 'Dataset A' in dname:
        info_type = 'Player attributes + performance + market value + transfers'
        relevance = 'High'
    elif 'Dataset B' in dname:
        info_type = 'Transfer transactions + market value at transfer'
        relevance = 'High'
    else:
        info_type = 'Modeling feature references / repository-level dataset context'
        relevance = 'Medium'

    comparison_rows.append({
        'Dataset': dname,
        'Key variables': ', '.join(key_vars) if key_vars else 'No direct market-value fields in this table',
        'Data size': f"{df.shape[0]:,} rows x {df.shape[1]:,} cols",
        'Type of information': info_type,
        'Relevance to project': relevance,
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)


# 8 Testing and Verification

Simple checks to verify profiling setup and loaded data.


In [ ]:
# Existence checks
assert DATASET_A_ROOT.exists(), 'Dataset A root path does not exist.'
assert DATASET_B_FILE.exists(), 'Dataset B transfer file is missing.'
assert DATASET_C_ROOT.exists(), 'Dataset C root path does not exist.'

print('Dataset paths verified successfully.')

# Loaded dataset checks
assert len(loaded_datasets) > 0, 'No datasets were loaded.'

for dname, payload in loaded_datasets.items():
    df = payload['df']
    assert df is not None, f'{dname} is not loaded.'
    assert len(df) > 0, f'{dname} has no rows.'
    print(f'{dname}: Dataset loaded successfully ({df.shape[0]} rows, {df.shape[1]} cols).')

# Expected column checks for key tables
assert {'player_id'}.issubset(set(loaded_datasets['Dataset A - player_profiles']['df'].columns))
assert {'player_id', 'value'}.issubset(set(loaded_datasets['Dataset A - player_market_value']['df'].columns))
assert {'player_id', 'transfer_fee_amnt', 'market_val_amnt'}.issubset(set(loaded_datasets['Dataset B - transfers']['df'].columns))

print('Expected columns verified for Dataset A and Dataset B.')

# Dataset C specific check
if any(name.startswith('Dataset C - file_') for name in loaded_datasets):
    print('Dataset C CSV files were found and loaded successfully.')
else:
    assert 'Dataset C - referenced_csv_inventory' in loaded_datasets, 'Dataset C fallback profiling table missing.'
    print('Dataset C local CSV files were not found; repository reference inventory created successfully.')

print('All profiling verification checks passed.')


# 9 Summary of Data Profiling

This profiling step indicates:

- Dataset A contains direct market value history and broad player/performance context.
- Dataset B contains transfer-centric variables including transfer fees and market value at transfer.
- Dataset C contributes modeling context and expected feature datasets (and local CSVs if available).

Together, these datasets are suitable for investigating market value drivers from player attributes, performance statistics, and transfer-market behavior.


In [ ]:
def contains_keywords(columns, keywords):
    cols_lower = [c.lower() for c in columns]
    return any(any(k in c for k in keywords) for c in cols_lower)

market_value_tables = []
transfer_tables = []
performance_tables = []

for dname, payload in loaded_datasets.items():
    cols = list(payload['df'].columns)
    if contains_keywords(cols, ['market_val', 'market_value', 'value']):
        market_value_tables.append(dname)
    if contains_keywords(cols, ['transfer', 'fee', 'window']):
        transfer_tables.append(dname)
    if contains_keywords(cols, ['goal', 'assist', 'minutes', 'clean_sheets', 'injury']):
        performance_tables.append(dname)

print('Tables with market value information:')
for t in market_value_tables:
    print('-', t)

print('\nTables with transfer information:')
for t in transfer_tables:
    print('-', t)

print('\nTables with performance/injury statistics:')
for t in performance_tables:
    print('-', t)

print('\nData profiling completed: Step 1 only (no structuring, no enrichment, no cleaning).')
